In [8]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl


reiteracio = ctrl.Antecedent(np.arange(0, 11, 1), 'reiteracio')
simptomes = ctrl.Antecedent(np.arange(0, 101, 1), 'simptomes')
aisllament = ctrl.Antecedent(np.arange(0, 11, 1), 'aisllament')
desequilibri = ctrl.Antecedent(np.arange(0, 11, 1), 'desequilibri')
ciber = ctrl.Antecedent(np.arange(0, 11, 1), 'ciberassetjament')

risc = ctrl.Consequent(np.arange(0, 11, 1), 'risc')

reiteracio['puntual'] = fuzz.trimf(reiteracio.universe, [0, 0, 2])
reiteracio['en_proces'] = fuzz.trimf(reiteracio.universe, [1, 2, 3])
reiteracio['sistematic'] = fuzz.trapmf(reiteracio.universe, [2, 4, 10, 10])

simptomes['lleu'] = fuzz.trapmf(simptomes.universe, [0, 0, 20, 40])
simptomes['moderat'] = fuzz.trimf(simptomes.universe, [20, 50, 80])
simptomes['sever'] = fuzz.trapmf(simptomes.universe, [60, 80, 100, 100])

aisllament['integrat'] = fuzz.trapmf(aisllament.universe, [0, 0, 3, 5])
aisllament['vulnerable'] = fuzz.trimf(aisllament.universe, [3, 5.5, 8])
aisllament['aillat'] = fuzz.trapmf(aisllament.universe, [6, 8, 10, 10])

desequilibri['nul'] = fuzz.trapmf(desequilibri.universe, [0, 0, 2, 4])
desequilibri['lleuger'] = fuzz.trimf(desequilibri.universe, [2, 4.5, 7])
desequilibri['evident'] = fuzz.trapmf(desequilibri.universe, [5, 8, 10, 10])

ciber['inexistent'] = fuzz.trimf(ciber.universe, [0, 0, 3])
ciber['limitat'] = fuzz.trimf(ciber.universe, [2, 5, 8])
ciber['viral'] = fuzz.trimf(ciber.universe, [7, 10, 10])

risc['sense_risc'] = fuzz.trapmf(risc.universe, [0, 0, 2, 4])
risc['risc_mig'] = fuzz.trimf(risc.universe, [2, 5, 8])
risc['risc_alt'] = fuzz.trapmf(risc.universe, [6, 8, 10, 10])

regles = [
    ctrl.Rule(reiteracio['puntual'] & simptomes['lleu'] & desequilibri['nul'], risc['sense_risc']),
    ctrl.Rule(reiteracio['puntual'] & aisllament['integrat'] & ciber['inexistent'], risc['sense_risc']),
    ctrl.Rule(reiteracio['en_proces'] & simptomes['lleu'] & desequilibri['nul'], risc['sense_risc']),
    ctrl.Rule(aisllament['integrat'] & ciber['inexistent'] & simptomes['lleu'], risc['sense_risc']),

    ctrl.Rule(reiteracio['en_proces'] & simptomes['moderat'] & desequilibri['lleuger'], risc['risc_mig']),
    ctrl.Rule(reiteracio['puntual'] & ciber['limitat'] & aisllament['vulnerable'], risc['risc_mig']),
    ctrl.Rule(reiteracio['en_proces'] & aisllament['vulnerable'] & simptomes['lleu'], risc['risc_mig']),
    ctrl.Rule(desequilibri['evident'] & reiteracio['puntual'] & simptomes['moderat'], risc['risc_mig']),

    ctrl.Rule(reiteracio['sistematic'] & desequilibri['evident'] & simptomes['sever'], risc['risc_alt']),
    ctrl.Rule(reiteracio['sistematic'] & aisllament['aillat'] & ciber['viral'], risc['risc_alt']),
    ctrl.Rule(ciber['viral'] & simptomes['sever'] & desequilibri['evident'], risc['risc_alt']),
    ctrl.Rule(reiteracio['sistematic'] & aisllament['aillat'] & simptomes['moderat'], risc['risc_alt'])
]

bullying_ctrl = ctrl.ControlSystem(regles)
simulador = ctrl.ControlSystemSimulation(bullying_ctrl)

def obtenir_entrada_validada(missatge, minim, maxim):
    while True:
        try:
            valor = float(input(missatge))
            if minim <= valor <= maxim:
                return valor
            print(f"Error: El valor ha d'estar entre {minim} i {maxim}.")
        except ValueError:
            print("Error: Introdueix un número vàlid.")

def executar_simulacio(reit, simp, aisll, deseq, cib, metode='centroid'):
    simulador.input['reiteracio'] = reit
    simulador.input['simptomes'] = simp
    simulador.input['aisllament'] = aisll
    simulador.input['desequilibri'] = deseq
    simulador.input['ciberassetjament'] = cib
    
    risc.defuzzify_method = metode
    
    try:
        simulador.compute()
        return simulador.output['risc']
    except KeyError:
        return None

print("\n--- PROVA EL TEU PROPI CAS ---")
u_reit = obtenir_entrada_validada("Reiteració (0-10): ", 0, 10)
u_simp = obtenir_entrada_validada("Símptomes (0-100): ", 0, 100)
u_aisll = obtenir_entrada_validada("Aïllament (0-10): ", 0, 10)
u_deseq = obtenir_entrada_validada("Desequilibri (0-10): ", 0, 10)
u_ciber = obtenir_entrada_validada("Ciberassetjament (0-10): ", 0, 10)

resultat = executar_simulacio(u_reit, u_simp, u_aisll, u_deseq, u_ciber)

if resultat is None:
    print("\n[!] ALERTA DE SISTEMA EXPERT:")
    print("Aquesta combinació de valors ha caigut en un 'punt cec' de la Base de Coneixements.")
    print("Cap de les regles definides s'ha activat. Caldria afegir més regles per cobrir aquest cas específic.")
else:
    print(f"\nEl nivell de risc calculat (Centroide) és: {resultat:.2f} sobre 10.")

    if resultat < 3.5:
        print("Conclusió: Situació sense risc aparent.")
    elif resultat < 6.5:
        print("Conclusió: Risc mitjà. Cal seguiment del tutor.")
    else:
        print("Conclusió: RISC ALT. Activar protocol d'assetjament immediatament.")


--- PROVA EL TEU PROPI CAS ---


Reiteració (0-10):  0
Símptomes (0-100):  0
Aïllament (0-10):  0
Desequilibri (0-10):  0
Ciberassetjament (0-10):  0



El nivell de risc calculat (Centroide) és: 1.56 sobre 10.
Conclusió: Situació sense risc aparent.
